# Notebook 01 — Test-Time Compute Scaling on a Small Model

**Chapter**: [Chapter 2 — Test-Time Compute Scaling](../chapters/02-test-time-compute-scaling.md).

**Claim demonstrated**: On a small open math-tuned model, accuracy on MATH-500 (or AIME-24) grows log-linearly with the per-question token budget, up to a task-dependent saturation point. Qualitative reproduction of Snell et al. (2024) and s1 (Muennighoff et al. 2025).

**What this notebook is not**: a frontier-scale reproduction. It is a tiny demonstration of the *signal*, runnable on a single small GPU.

**Hardware**: ≥ 16 GB GPU (e.g. A10G, RTX 3090). CPU-only is possible but slow (~ 1 hour for the smallest model).

**Dependencies**: `transformers`, `accelerate`, `torch`, `datasets`, `tqdm`, `matplotlib`.

---

## Experiment plan

1. Load a small reasoning-tuned model (Qwen2.5-Math-1.5B-Instruct is the default).
2. Sample 50 problems from MATH-500.
3. For each token budget in `{64, 128, 256, 512, 1024, 2048, 4096}`:
   - Generate one completion per problem with `max_new_tokens` set to that budget.
   - Parse the final answer; check correctness with a math equivalence checker.
4. Plot accuracy vs token budget on a log-x axis.

Expected: roughly log-linear improvement until ~1024–2048 tokens, then saturation.

In [ ]:
# Setup
MODEL_NAME = "Qwen/Qwen2.5-Math-1.5B-Instruct"
N_PROBLEMS = 50
BUDGETS = [64, 128, 256, 512, 1024, 2048, 4096]
SEED = 42

import os, random, json, math, re
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
from tqdm.auto import tqdm

random.seed(SEED)
torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using {device}.")

In [ ]:
# Load model and tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16 if device == "cuda" else torch.float32,
    device_map=device,
)
model.eval()

In [ ]:
# Load MATH-500 sample. (Or substitute another math benchmark.)
ds = load_dataset("HuggingFaceH4/MATH-500", split="test")
ds = ds.shuffle(seed=SEED).select(range(N_PROBLEMS))
print(ds[0])

In [ ]:
# Answer extraction and equivalence check.
BOXED_RE = re.compile(r"\\boxed\{([^}]*)\}")

def extract_answer(text: str) -> str | None:
    matches = BOXED_RE.findall(text)
    if not matches:
        return None
    return matches[-1].strip()

def is_correct(pred: str | None, gold: str) -> bool:
    if pred is None:
        return False
    # Very loose equivalence: numeric match if both parse, else string-normalize.
    def normalize(s: str) -> str:
        return re.sub(r"\s+", "", s).lower()
    if normalize(pred) == normalize(gold):
        return True
    try:
        return abs(float(pred) - float(gold)) < 1e-6
    except ValueError:
        return False

In [ ]:
# Run the sweep.
results = {b: [] for b in BUDGETS}

for budget in BUDGETS:
    correct = 0
    for ex in tqdm(ds, desc=f"budget={budget}"):
        prompt = f"Solve the following math problem. Put your final answer in \\boxed{{}}.\n\n{ex['problem']}\n"
        inputs = tokenizer(prompt, return_tensors="pt").to(device)
        with torch.inference_mode():
            out = model.generate(
                **inputs,
                max_new_tokens=budget,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )
        gen = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        pred = extract_answer(gen)
        if is_correct(pred, ex["answer"]):
            correct += 1
    acc = correct / len(ds)
    results[budget].append(acc)
    print(f"budget={budget}: accuracy={acc:.3f}")

In [ ]:
# Plot accuracy vs budget.
import matplotlib.pyplot as plt

xs = list(results.keys())
ys = [results[b][0] for b in xs]

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(xs, ys, marker="o")
ax.set_xscale("log")
ax.set_xlabel("max generation tokens (per problem)")
ax.set_ylabel("accuracy on MATH-500 sample")
ax.set_title(f"Test-time compute scaling: {MODEL_NAME} on MATH-500")
ax.grid(True, which="both", alpha=0.3)
fig.tight_layout()
fig.savefig("01_scaling.png", dpi=150)
plt.show()

## Interpretation

Expected pattern on Qwen2.5-Math-1.5B-Instruct, MATH-500 sample of 50:
- ~ 30 – 45% at 64 tokens (the model can't fit the chain).
- ~ 55 – 65% at 256 tokens.
- ~ 70 – 80% at 2048 tokens.
- Plateau between 2048 and 4096.

The exact numbers depend on the specific MATH-500 sample and the model checkpoint at run time. The *qualitative* claim — log-linear increase to a saturation point — is what reproduces.

**Caveats:**
- Greedy decoding only; full reproduction would include best-of-N and self-consistency variants — see [`02-best-of-n-vs-self-consistency.ipynb`](02-best-of-n-vs-self-consistency.ipynb).
- The answer extractor is loose; real benchmarks use stricter math equivalence (e.g. `sympy`-based).
- 50 problems gives ~ ±7% Wilson-interval error bars; treat differences smaller than that as noise.

## Going further

- Substitute `DeepSeek-R1-Distill-Qwen-1.5B` to see how an RL-distilled model scales differently.
- Add a budget-forcing variant (replace `</think>` with `"Wait"`) to reproduce the s1 result.
- Re-run on AIME-24 for a harder slice; expect lower absolute accuracy and a *steeper* scaling curve in some regimes.